# Radical-Aligned Structure — Colab T4 fast run (8 models + cloze)

Free Colab T4. Total runtime ~75 minutes.

Before running:
1. Runtime → Change runtime type → **T4 GPU**.
2. Run cells top-to-bottom.

When done, cell 13 packages `results.zip`. Cell 14 downloads it via the browser.

In [ ]:
# 1. Clone the repo
!git clone https://github.com/aryan35790jp/chinese_llm.git /content/repo
%cd /content/repo

In [ ]:
# 2. Install dependency stack. Colab ships torch + CUDA preinstalled.
!pip install -q transformers==4.44.2 tokenizers==0.19.1 \
    numpy==1.26.4 pandas==2.2.2 scipy==1.13.1 scikit-learn==1.5.1 \
    matplotlib==3.9.0 tqdm==4.66.4 datasets==2.20.0 sentencepiece==0.2.0 \
    Pillow==10.4.0 umap-learn==0.5.6 \
    fugashi==1.3.2 unidic-lite==1.0.8 ipadic==1.0.0
# The dependency-conflict warnings about numpy/scipy from other Colab packages are SAFE to ignore.

In [ ]:
# 3. Confirm GPU
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1))

In [ ]:
# 4. Smoke tests (~30s)
!python scripts/new/_check_syntax.py
!python scripts/new/_smoke_test.py

In [ ]:
# 5. Build dataset (~3 min)
!mkdir -p data/unihan
!curl -sL -o data/unihan/Unihan.zip https://www.unicode.org/Public/UCD/latest/ucd/Unihan.zip
!cd data/unihan && unzip -o -q Unihan.zip && cd ../..
!python scripts/new/dataset_builder.py
!python scripts/new/classify_liushu.py

In [ ]:
# 6. Tokenization audit (~2 min) — appendix table
import os
os.environ['RADICAL_FAST'] = '1'
!python scripts/new/tokenization_audit.py

In [ ]:
# 7. THE HEAVY STEP: extract embeddings.
# 8 models (mBERT, Chinese-BERT, XLM-R-base, JP-BERT-char, Qwen2.5-1.5B,
# Qwen2.5-3B, BGE-large-zh, plus the local glyph baseline).
# Total: ~45 min on free T4. The two Qwen LLMs are the slow part (~15 min combined).
import os
os.environ['RADICAL_FAST'] = '1'
!RADICAL_FAST=1 python scripts/new/extract_embeddings.py

In [ ]:
# 8. Pure-vision baseline (~3 min)
!apt-get install -y -q fonts-noto-cjk 2>/dev/null || true
!python scripts/new/glyph_only_baseline.py

In [ ]:
# 9. Isotropy correction (~3 min)
!python scripts/new/isotropy_correction.py

In [ ]:
# 10. All cheap analysis steps (~15 min). Skips the slow Wikipedia stream.
import os
os.environ['RADICAL_FAST'] = '1'
!RADICAL_FAST=1 python scripts/new/full_pipeline.py --fast --skip extract,glyph_baseline,isotropy

In [ ]:
# 11. Variance decomposition (~25 min — Wikipedia stream first run, then cached)
import os
os.environ['RADICAL_FAST'] = '1'
!RADICAL_FAST=1 python scripts/new/cooccurrence_baseline.py

In [ ]:
# 12. Radical cloze probe (~15 min) — the downstream LM-behavior task.
# Scores radical-aware models vs radical-naive ones on cloze trials.
!python scripts/new/radical_cloze_probe.py

In [ ]:
# 13. Regenerate report and figures with the cloze + variance results
!python scripts/new/figures.py
!python scripts/new/results_report.py
with open('results/_REPORT.md', encoding='utf-8') as f:
    from IPython.display import Markdown
    md = f.read()
Markdown(md)

In [ ]:
# 14. Pack and download
!cd /content/repo && zip -qr /content/results.zip results figures data/radical_dataset.csv data/radical_summary.csv
from google.colab import files
files.download('/content/results.zip')